# makemore: part 3


In [1]:
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda' and hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

SEED = 2147483647
LONG_RUN_STEPS = 200_000  # lower this for a quick run
DEBUG_STEPS = 1_000
BATCH_SIZE = 32

print('torch:', torch.__version__)
print('device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(device))
print('long training steps:', f'{LONG_RUN_STEPS:,}')


torch: 2.12.1+cu130
device: cuda
GPU: NVIDIA GeForce RTX 5090 Laptop GPU
long training steps: 200,000


In [2]:
# find names.txt from the notebook directory or the repository root
data_candidates = [
    Path('names.txt'),
    Path('book/04-makemore-batchnorm/names.txt'),
]
DATA_PATH = next((p for p in data_candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('names.txt not found; start from the notebook directory or the repository root')

words = DATA_PATH.read_text().splitlines()
print(words[:8])
print('number of words:', len(words))


['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']
number of words: 32033


In [3]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(set(''.join(words)))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print('vocab_size:', vocab_size)


{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
vocab_size: 27


In [4]:
block_size = 3  # context length: how many characters do we take to predict the next one?

def build_dataset(word_subset):
    X, Y = [], []
    for w in word_subset:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    # build once, then keep the dataset on the device
    X = torch.tensor(X, dtype=torch.long, device=device)
    Y = torch.tensor(Y, dtype=torch.long, device=device)
    print(tuple(X.shape), tuple(Y.shape), X.device)
    return X, Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr,  Ytr  = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte,  Yte  = build_dataset(words[n2:])


(182625, 3) (182625,) cuda:0
(22655, 3) (22655,) cuda:0
(22866, 3) (22866,) cuda:0


In [5]:
# initialize on CPU with the seeded generator, then move to the device
# generate the long training batches directly on the device
def cpu_generator(seed=SEED):
    return torch.Generator().manual_seed(seed)

def randn_on_device(shape, generator, scale=1.0):
    return (torch.randn(shape, generator=generator) * scale).to(device)

def one_video_batch(generator, batch_size=BATCH_SIZE):
    # use the initialization generator for the first diagnostic batch
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=generator)
    ix = ix.to(device)
    return Xtr[ix], Ytr[ix]

def prefetch_training_batches(max_steps, batch_size=BATCH_SIZE, seed=SEED + 1):
    # preload all training batches to avoid two indexing kernels per step
    # this uses about 205 MB at 200k steps and batch size 32
    generator_device = 'cuda' if device.type == 'cuda' else 'cpu'
    bg = torch.Generator(device=generator_device).manual_seed(seed)
    indices = torch.randint(
        0, Xtr.shape[0], (max_steps, batch_size),
        generator=bg, device=device,
    )
    return Xtr[indices], Ytr[indices]

def sync_device():
    if device.type == 'cuda':
        torch.cuda.synchronize()

def plot_loss_history(loss_history, title):
    values = loss_history.detach().float().cpu()
    plt.figure(figsize=(8, 3))
    plt.plot(values)
    plt.xlabel('step')
    plt.ylabel('log10(loss)')
    plt.title(title)
    plt.show()

@torch.no_grad()
def evaluate_plain(C, W1, b1, W2, b2):
    for name, x, y in (
        ('train', Xtr, Ytr),
        ('val', Xdev, Ydev),
        ('test', Xte, Yte),
    ):
        emb = C[x]
        h = torch.tanh(emb.view(emb.shape[0], -1) @ W1 + b1)
        loss = F.cross_entropy(h @ W2 + b2, y)
        print(f'{name:>5s}: {loss.item():.6f}')

@torch.no_grad()
def sample_plain(C, W1, b1, W2, b2, count=20):
    generator_device = 'cuda' if device.type == 'cuda' else 'cpu'
    sg = torch.Generator(device=generator_device).manual_seed(SEED + 10)
    samples = []
    for _ in range(count):
        out = []
        context = [0] * block_size
        while True:
            x = torch.tensor([context], dtype=torch.long, device=device)
            emb = C[x]
            h = torch.tanh(emb.view(1, -1) @ W1 + b1)
            probs = F.softmax(h @ W2 + b2, dim=1)
            ix = torch.multinomial(probs, 1, generator=sg).item()
            context = context[1:] + [ix]
            out.append(ix)
            if ix == 0:
                break
        samples.append(''.join(itos[i] for i in out))
    print('\n'.join(samples))


In [ ]:
# MLP revisited
n_embd = 10
n_hidden = 200

g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g)
b1 = randn_on_device((n_hidden,), g)
W2 = randn_on_device((n_hidden, vocab_size), g)
b2 = randn_on_device((vocab_size,), g)

parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad_(True)
print('parameter count:', sum(p.numel() for p in parameters))


In [ ]:
# same optimization as last time
X_batches, Y_batches = prefetch_training_batches(LONG_RUN_STEPS)
loss_history = torch.empty(LONG_RUN_STEPS, device=device)
sync_device()
started = time.perf_counter()

for i in range(LONG_RUN_STEPS):
    Xb, Yb = X_batches[i], Y_batches[i]

    emb = C[Xb]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1 if i < LONG_RUN_STEPS // 2 else 0.01
    with torch.no_grad():
        for p in parameters:
            p.add_(p.grad, alpha=-lr)
        loss_history[i] = loss.log10()

    if i % max(1, LONG_RUN_STEPS // 20) == 0:
        print(f'{i:7d}/{LONG_RUN_STEPS:7d}: {loss.item():.4f}')

sync_device()
print(f'elapsed: {time.perf_counter() - started:.1f}s on {device}')
del X_batches, Y_batches


In [ ]:
plot_loss_history(loss_history, 'original initialization')
evaluate_plain(C, W1, b1, W2, b2)


In [ ]:
# sample from the model
sample_plain(C, W1, b1, W2, b2)


## fixing the initial loss

At initialization we expect a uniform distribution over 27 characters, so the loss should be `-log(1/27)`. Stop after the first iteration and inspect the logits.


In [ ]:
expected_initial_loss = -math.log(1.0 / vocab_size)
print('expected initial loss:', expected_initial_loss)


In [ ]:
# a four-class example
toy_g = cpu_generator(SEED + 123)
toy_logits = {
    'all zero': torch.zeros(4),
    'small random': torch.randn(4, generator=toy_g),
    '10x random': torch.randn(4, generator=toy_g) * 10,
    '100x random': torch.randn(4, generator=toy_g) * 100,
}
target = 2
for name, logits_cpu in toy_logits.items():
    probs = F.softmax(logits_cpu, dim=0)
    loss = -probs[target].log()
    print(f'{name:>12s} | logits={logits_cpu.tolist()}')
    print(f'{"":>12s} | probs ={probs.tolist()} | loss={loss.item():.4f}')


In [ ]:
# stop after the first iteration and inspect the logits
g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g)
b1 = randn_on_device((n_hidden,), g)
W2 = randn_on_device((n_hidden, vocab_size), g)
b2 = randn_on_device((vocab_size,), g)

Xb, Yb = one_video_batch(g)
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)
hpreact = embcat @ W1 + b1
h = torch.tanh(hpreact)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Yb)

print('first loss:', loss.item())
print('logits[0]:', logits[0].detach().cpu())
print('logit mean/std/min/max:', *(f'{x:.3f}' for x in (
    logits.mean().item(), logits.std().item(), logits.min().item(), logits.max().item()
)))


In [ ]:
# reset from the same seed
g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g)
b1 = randn_on_device((n_hidden,), g)
W2 = randn_on_device((n_hidden, vocab_size), g, scale=0.1)
b2 = randn_on_device((vocab_size,), g, scale=0.0)

Xb, Yb = one_video_batch(g)
embcat = C[Xb].view(BATCH_SIZE, -1)
hpreact = embcat @ W1 + b1
h = torch.tanh(hpreact)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Yb)
print('W2 scale=0.1, b2=0 -> first loss:', loss.item())
print('logit std/min/max:', logits.std().item(), logits.min().item(), logits.max().item())


In [ ]:
g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g)
b1 = randn_on_device((n_hidden,), g)
W2 = randn_on_device((n_hidden, vocab_size), g, scale=0.01)
b2 = randn_on_device((vocab_size,), g, scale=0.0)

Xb, Yb = one_video_batch(g)
embcat = C[Xb].view(BATCH_SIZE, -1)
hpreact = embcat @ W1 + b1
h = torch.tanh(hpreact)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Yb)
print('expected:', expected_initial_loss)
print('actual:  ', loss.item())
print('logit std/min/max:', logits.std().item(), logits.min().item(), logits.max().item())


In [ ]:
g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g)
b1 = randn_on_device((n_hidden,), g)
W2 = randn_on_device((n_hidden, vocab_size), g, scale=0.01)
b2 = randn_on_device((vocab_size,), g, scale=0.0)
parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad_(True)

X_batches, Y_batches = prefetch_training_batches(LONG_RUN_STEPS)
loss_history = torch.empty(LONG_RUN_STEPS, device=device)
sync_device(); started = time.perf_counter()

for i in range(LONG_RUN_STEPS):
    Xb, Yb = X_batches[i], Y_batches[i]
    embcat = C[Xb].view(BATCH_SIZE, -1)
    hpreact = embcat @ W1 + b1
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < LONG_RUN_STEPS // 2 else 0.01
    with torch.no_grad():
        for p in parameters:
            p.add_(p.grad, alpha=-lr)
        loss_history[i] = loss.log10()
    if i % max(1, LONG_RUN_STEPS // 20) == 0:
        print(f'{i:7d}/{LONG_RUN_STEPS:7d}: {loss.item():.4f}')

sync_device(); print(f'elapsed: {time.perf_counter() - started:.1f}s on {device}')
del X_batches, Y_batches


In [ ]:
plot_loss_history(loss_history, 'fixed output initialization (W2 x 0.01, b2 = 0)')
evaluate_plain(C, W1, b1, W2, b2)


## fixing the saturated tanh

The logits now look good, but many hidden activations are near -1 or 1. Stop after one batch and inspect `h` and `hpreact`.


In [ ]:
g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g)
b1 = randn_on_device((n_hidden,), g)
W2 = randn_on_device((n_hidden, vocab_size), g, scale=0.01)
b2 = randn_on_device((vocab_size,), g, scale=0.0)

Xb, Yb = one_video_batch(g)
embcat = C[Xb].view(BATCH_SIZE, -1)
hpreact = embcat @ W1 + b1
h = torch.tanh(hpreact)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Yb)

print('first loss:', loss.item())
print('h shape:', tuple(h.shape))
print('h[0, :30]:', h[0, :30].detach().cpu())


In [ ]:
# visualize the hidden activations
plt.figure(figsize=(8, 3))
plt.hist(h.detach().flatten().cpu().tolist(), bins=50)
plt.title('h = tanh(hpreact), original W1/b1')
plt.xlabel('activation')
plt.show()


In [ ]:
# visualize the pre-activations
plt.figure(figsize=(8, 3))
plt.hist(hpreact.detach().flatten().cpu().tolist(), bins=50)
plt.title('hpreact, original W1/b1')
plt.xlabel('pre-activation')
plt.show()
print('hpreact mean/std/min/max:', hpreact.mean().item(), hpreact.std().item(), hpreact.min().item(), hpreact.max().item())


In [ ]:
tanh_local_grad = 1 - h.detach() ** 2
saturated = h.detach().abs() > 0.99

print('saturated fraction:', saturated.float().mean().item())
print('mean local derivative:', tanh_local_grad.mean().item())

plt.figure(figsize=(12, 4))
plt.imshow(saturated.cpu(), cmap='gray', interpolation='nearest', aspect='auto')
plt.xlabel('hidden neuron')
plt.ylabel('batch example')
plt.title('white: |h| > 0.99 (gradient is almost stopped)')
plt.show()

fully_saturated_columns = saturated.all(dim=0).sum().item()
print('fully saturated neurons in this batch:', fully_saturated_columns, '/', n_hidden)


In [ ]:
g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g, scale=0.1)
b1 = randn_on_device((n_hidden,), g, scale=0.01)
W2 = randn_on_device((n_hidden, vocab_size), g, scale=0.01)
b2 = randn_on_device((vocab_size,), g, scale=0.0)

Xb, Yb = one_video_batch(g)
embcat = C[Xb].view(BATCH_SIZE, -1)
hpreact = embcat @ W1 + b1
h = torch.tanh(hpreact)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(hpreact.detach().flatten().cpu().tolist(), bins=50)
axes[0].set_title('hpreact: W1 x 0.1, b1 x 0.01')
axes[1].hist(h.detach().flatten().cpu().tolist(), bins=50)
axes[1].set_title('h = tanh(hpreact)')
plt.show()
print('h std:', h.std().item(), 'saturated:', (h.abs() > 0.99).float().mean().item())


In [ ]:
g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g, scale=0.2)
b1 = randn_on_device((n_hidden,), g, scale=0.01)
W2 = randn_on_device((n_hidden, vocab_size), g, scale=0.01)
b2 = randn_on_device((vocab_size,), g, scale=0.0)

Xb, Yb = one_video_batch(g)
embcat = C[Xb].view(BATCH_SIZE, -1)
hpreact = embcat @ W1 + b1
h = torch.tanh(hpreact)

fig, axes = plt.subplots(1, 3, figsize=(16, 3))
axes[0].hist(hpreact.detach().flatten().cpu().tolist(), bins=50)
axes[0].set_title('hpreact: W1 x 0.2')
axes[1].hist(h.detach().flatten().cpu().tolist(), bins=50)
axes[1].set_title('h')
axes[2].imshow((h.detach().abs() > 0.99).cpu(), cmap='gray', aspect='auto')
axes[2].set_title('|h| > 0.99')
plt.show()
print('h std:', h.std().item(), 'saturated:', (h.abs() > 0.99).float().mean().item())


In [ ]:
g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g, scale=0.2)
b1 = randn_on_device((n_hidden,), g, scale=0.01)
W2 = randn_on_device((n_hidden, vocab_size), g, scale=0.01)
b2 = randn_on_device((vocab_size,), g, scale=0.0)
parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad_(True)

X_batches, Y_batches = prefetch_training_batches(LONG_RUN_STEPS)
loss_history = torch.empty(LONG_RUN_STEPS, device=device)
sync_device(); started = time.perf_counter()

for i in range(LONG_RUN_STEPS):
    Xb, Yb = X_batches[i], Y_batches[i]
    embcat = C[Xb].view(BATCH_SIZE, -1)
    hpreact = embcat @ W1 + b1
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)
    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < LONG_RUN_STEPS // 2 else 0.01
    with torch.no_grad():
        for p in parameters:
            p.add_(p.grad, alpha=-lr)
        loss_history[i] = loss.log10()
    if i % max(1, LONG_RUN_STEPS // 20) == 0:
        print(f'{i:7d}/{LONG_RUN_STEPS:7d}: {loss.item():.4f}')

sync_device(); print(f'elapsed: {time.perf_counter() - started:.1f}s on {device}')
del X_batches, Y_batches


In [ ]:
plot_loss_history(loss_history, 'fixed output + less saturated tanh')
evaluate_plain(C, W1, b1, W2, b2)


## semi-principled initialization

The `0.2` scale is a magic number. Use `gain / sqrt(fan_in)` instead.


In [ ]:
demo_g = cpu_generator(SEED + 200)
x = randn_on_device((1000, 10), demo_g)
w = randn_on_device((10, 200), demo_g)
y = x @ w
print('x mean/std:', x.mean().item(), x.std().item())
print('y mean/std:', y.mean().item(), y.std().item())

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(x.detach().flatten().cpu().tolist(), bins=50, density=True)
axes[0].set_title('x')
axes[1].hist(y.detach().flatten().cpu().tolist(), bins=50, density=True)
axes[1].set_title('y = x @ w (unscaled)')
plt.show()


In [ ]:
# compare a few weight scales
variants = {
    'w x 5': w * 5,
    'w x 0.2': w * 0.2,
    'w / sqrt(10)': w / math.sqrt(10),
}
fig, axes = plt.subplots(1, 3, figsize=(16, 3))
for ax, (name, scaled_w) in zip(axes, variants.items()):
    out = x @ scaled_w
    ax.hist(out.detach().flatten().cpu().tolist(), bins=50, density=True)
    ax.set_title(f'{name}\nstd={out.std().item():.3f}')
plt.show()


In [ ]:
fan_in = n_embd * block_size
tanh_gain = 5 / 3
kaiming_scale = tanh_gain / math.sqrt(fan_in)
print('fan_in:', fan_in)
print('tanh gain:', tanh_gain)
print('recommended W1 std:', kaiming_scale)
print('torch calculate_gain(tanh):', torch.nn.init.calculate_gain('tanh'))


In [ ]:
g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g, scale=kaiming_scale)
b1 = randn_on_device((n_hidden,), g, scale=0.01)
W2 = randn_on_device((n_hidden, vocab_size), g, scale=0.01)
b2 = randn_on_device((vocab_size,), g, scale=0.0)
parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad_(True)

X_batches, Y_batches = prefetch_training_batches(LONG_RUN_STEPS)
loss_history = torch.empty(LONG_RUN_STEPS, device=device)
sync_device(); started = time.perf_counter()

for i in range(LONG_RUN_STEPS):
    Xb, Yb = X_batches[i], Y_batches[i]
    embcat = C[Xb].view(BATCH_SIZE, -1)
    hpreact = embcat @ W1 + b1
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)
    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < LONG_RUN_STEPS // 2 else 0.01
    with torch.no_grad():
        for p in parameters:
            p.add_(p.grad, alpha=-lr)
        loss_history[i] = loss.log10()
    if i % max(1, LONG_RUN_STEPS // 20) == 0:
        print(f'{i:7d}/{LONG_RUN_STEPS:7d}: {loss.item():.4f}')

sync_device(); print(f'elapsed: {time.perf_counter() - started:.1f}s on {device}')
del X_batches, Y_batches


In [ ]:
plot_loss_history(loss_history, 'Kaiming-style W1 initialization')
evaluate_plain(C, W1, b1, W2, b2)


## batch normalization

Normalize each hidden unit over the batch, then learn a scale and shift.


In [ ]:
g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g, scale=kaiming_scale)
b1 = randn_on_device((n_hidden,), g, scale=0.01)
W2 = randn_on_device((n_hidden, vocab_size), g, scale=0.01)
b2 = randn_on_device((vocab_size,), g, scale=0.0)

Xb, Yb = one_video_batch(g)
embcat = C[Xb].view(BATCH_SIZE, -1)
hpreact_raw = embcat @ W1 + b1
bnmeani = hpreact_raw.mean(0, keepdim=True)
bnstdi = hpreact_raw.std(0, keepdim=True)
hpreact_normalized = (hpreact_raw - bnmeani) / bnstdi

print('raw global mean/std:', hpreact_raw.mean().item(), hpreact_raw.std().item())
print('normalized per-neuron mean max abs:', hpreact_normalized.mean(0).abs().max().item())
print('normalized per-neuron std range:', hpreact_normalized.std(0).min().item(), hpreact_normalized.std(0).max().item())

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(hpreact_raw.detach().flatten().cpu().tolist(), bins=50)
axes[0].set_title('before BatchNorm')
axes[1].hist(hpreact_normalized.detach().flatten().cpu().tolist(), bins=50)
axes[1].set_title('after standardization')
plt.show()


In [ ]:
bngain = torch.ones((1, n_hidden), device=device)
bnbias = torch.zeros((1, n_hidden), device=device)
hpreact = bngain * hpreact_normalized + bnbias
h = torch.tanh(hpreact)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Yb)
print('first loss with BatchNorm affine parameters:', loss.item())
print('bngain/bnbias shapes:', tuple(bngain.shape), tuple(bnbias.shape))


In [ ]:
g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g, scale=kaiming_scale)
b1 = randn_on_device((n_hidden,), g, scale=0.01)
W2 = randn_on_device((n_hidden, vocab_size), g, scale=0.01)
b2 = randn_on_device((vocab_size,), g, scale=0.0)
bngain = torch.ones((1, n_hidden), device=device)
bnbias = torch.zeros((1, n_hidden), device=device)
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad_(True)

X_batches, Y_batches = prefetch_training_batches(LONG_RUN_STEPS)
loss_history = torch.empty(LONG_RUN_STEPS, device=device)
sync_device(); started = time.perf_counter()

for i in range(LONG_RUN_STEPS):
    Xb, Yb = X_batches[i], Y_batches[i]
    embcat = C[Xb].view(BATCH_SIZE, -1)
    hpreact = embcat @ W1 + b1
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < LONG_RUN_STEPS // 2 else 0.01
    with torch.no_grad():
        for p in parameters:
            p.add_(p.grad, alpha=-lr)
        loss_history[i] = loss.log10()
    if i % max(1, LONG_RUN_STEPS // 20) == 0:
        print(f'{i:7d}/{LONG_RUN_STEPS:7d}: {loss.item():.4f}')

sync_device(); print(f'elapsed: {time.perf_counter() - started:.1f}s on {device}')
del X_batches, Y_batches


In [ ]:
plot_loss_history(loss_history, 'BatchNorm using current-batch statistics')

@torch.no_grad()
def evaluate_bn_with_split_statistics():
    # this still uses statistics from the current split
    for name, x, y in (('train', Xtr, Ytr), ('val', Xdev, Ydev)):
        embcat = C[x].view(x.shape[0], -1)
        hp = embcat @ W1 + b1
        hp = bngain * (hp - hp.mean(0, keepdim=True)) / hp.std(0, keepdim=True) + bnbias
        logits = torch.tanh(hp) @ W2 + b2
        print(name, F.cross_entropy(logits, y).item())

evaluate_bn_with_split_statistics()


## batch norm at inference

First calibrate fixed statistics over the training set. Then keep running statistics during training.


In [ ]:
# calibrate the batch norm at the end of training
with torch.no_grad():
    embcat_all = C[Xtr].view(Xtr.shape[0], -1)
    hpreact_all = embcat_all @ W1 + b1
    bnmean = hpreact_all.mean(0, keepdim=True)
    bnstd = hpreact_all.std(0, keepdim=True)

@torch.no_grad()
def evaluate_bn_fixed_stats(mean, std):
    for name, x, y in (('train', Xtr, Ytr), ('val', Xdev, Ydev), ('test', Xte, Yte)):
        embcat = C[x].view(x.shape[0], -1)
        hp = embcat @ W1 + b1
        hp = bngain * (hp - mean) / std + bnbias
        logits = torch.tanh(hp) @ W2 + b2
        print(f'{name:>5s}: {F.cross_entropy(logits, y).item():.6f}')

evaluate_bn_fixed_stats(bnmean, bnstd)

# fixed statistics also allow single-example inference
one_context = Xdev[:1]
with torch.no_grad():
    hp = C[one_context].view(1, -1) @ W1 + b1
    hp = bngain * (hp - bnmean) / bnstd + bnbias
    one_logits = torch.tanh(hp) @ W2 + b2
print('single-example logits shape:', tuple(one_logits.shape))


In [ ]:
g = cpu_generator()
C  = randn_on_device((vocab_size, n_embd), g)
W1 = randn_on_device((n_embd * block_size, n_hidden), g, scale=kaiming_scale)
b1 = randn_on_device((n_hidden,), g, scale=0.01)
W2 = randn_on_device((n_hidden, vocab_size), g, scale=0.01)
b2 = randn_on_device((vocab_size,), g, scale=0.0)
bngain = torch.ones((1, n_hidden), device=device)
bnbias = torch.zeros((1, n_hidden), device=device)
bnmean_running = torch.zeros((1, n_hidden), device=device)
bnstd_running = torch.ones((1, n_hidden), device=device)
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad_(True)

X_batches, Y_batches = prefetch_training_batches(LONG_RUN_STEPS)
loss_history = torch.empty(LONG_RUN_STEPS, device=device)
sync_device(); started = time.perf_counter()

for i in range(LONG_RUN_STEPS):
    Xb, Yb = X_batches[i], Y_batches[i]
    embcat = C[Xb].view(BATCH_SIZE, -1)
    hpreact = embcat @ W1 + b1
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < LONG_RUN_STEPS // 2 else 0.01
    with torch.no_grad():
        for p in parameters:
            p.add_(p.grad, alpha=-lr)
        bnmean_running.mul_(0.999).add_(bnmeani, alpha=0.001)
        bnstd_running.mul_(0.999).add_(bnstdi, alpha=0.001)
        loss_history[i] = loss.log10()
    if i % max(1, LONG_RUN_STEPS // 20) == 0:
        print(f'{i:7d}/{LONG_RUN_STEPS:7d}: {loss.item():.4f}')

sync_device(); print(f'elapsed: {time.perf_counter() - started:.1f}s on {device}')
del X_batches, Y_batches


In [ ]:
# compare exact and running statistics
with torch.no_grad():
    embcat_all = C[Xtr].view(Xtr.shape[0], -1)
    hpreact_all = embcat_all @ W1 + b1
    bnmean_exact = hpreact_all.mean(0, keepdim=True)
    bnstd_exact = hpreact_all.std(0, keepdim=True)

print('mean: exact vs running (first 8)')
print(torch.stack((bnmean_exact[0, :8], bnmean_running[0, :8])).cpu())
print('std: exact vs running (first 8)')
print(torch.stack((bnstd_exact[0, :8], bnstd_running[0, :8])).cpu())
print('mean MAE:', (bnmean_exact - bnmean_running).abs().mean().item())
print('std MAE: ', (bnstd_exact - bnstd_running).abs().mean().item())


In [ ]:
@torch.no_grad()
def evaluate_bn_running_stats():
    for name, x, y in (('train', Xtr, Ytr), ('val', Xdev, Ydev), ('test', Xte, Yte)):
        embcat = C[x].view(x.shape[0], -1)
        hp = embcat @ W1 + b1
        hp = bngain * (hp - bnmean_running) / bnstd_running + bnbias
        logits = torch.tanh(hp) @ W2 + b2
        print(f'{name:>5s}: {F.cross_entropy(logits, y).item():.6f}')

plot_loss_history(loss_history, 'BatchNorm + running statistics')
evaluate_bn_running_stats()


In [ ]:
# b1 is canceled by the batch mean
print('b1 grad mean/std/maxabs:', b1.grad.mean().item(), b1.grad.std().item(), b1.grad.abs().max().item())

with torch.no_grad():
    embcat = C[Xtr[:BATCH_SIZE]].view(BATCH_SIZE, -1)
    raw_without_bias = embcat @ W1
    raw_with_bias = raw_without_bias + b1

    norm_without_bias = (
        raw_without_bias - raw_without_bias.mean(0, keepdim=True)
    ) / raw_without_bias.std(0, keepdim=True)
    norm_with_bias = (
        raw_with_bias - raw_with_bias.mean(0, keepdim=True)
    ) / raw_with_bias.std(0, keepdim=True)

    # use variance and add eps inside the square root
    xmean = raw_without_bias.mean(0, keepdim=True)
    xvar = raw_without_bias.var(0, keepdim=True)
    norm_with_eps = (raw_without_bias - xmean) / torch.sqrt(xvar + 1e-5)

print('max |BN(x+b1) - BN(x)|:', (norm_with_bias - norm_without_bias).abs().max().item())
print('finite with eps:', torch.isfinite(norm_with_eps).all().item())


In [ ]:
import torch.nn as nn

pytorch_motif = nn.Sequential(
    nn.Linear(n_embd * block_size, n_hidden, bias=False),
    nn.BatchNorm1d(n_hidden),
    nn.Tanh(),
).to(device)
print(pytorch_motif)
print('parameters:', [name for name, _ in pytorch_motif.named_parameters()])
print('buffers:   ', [name for name, _ in pytorch_motif.named_buffers()])


## summary + pytorchifying


In [ ]:
class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.fan_in = fan_in
        self.weight = randn_on_device((fan_in, fan_out), g, scale=fan_in ** -0.5)
        self.bias = torch.zeros(fan_out, device=device) if bias else None
        self.out = None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out = self.out + self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        # parameters (trained with backprop)
        self.gamma = torch.ones(dim, device=device)
        self.beta = torch.zeros(dim, device=device)
        # buffers (trained with a running momentum update)
        self.running_mean = torch.zeros(dim, device=device)
        self.running_var = torch.ones(dim, device=device)
        self.out = None

    def __call__(self, x):
        if self.training:
            xmean = x.mean(0)
            xvar = x.var(0)
        else:
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        if self.training:
            with torch.no_grad():
                self.running_mean.mul_(1 - self.momentum).add_(xmean, alpha=self.momentum)
                self.running_var.mul_(1 - self.momentum).add_(xvar, alpha=self.momentum)
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


class Tanh:
    def __init__(self):
        self.out = None

    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []


In [ ]:
# helpers for the deeper network
def build_deep_network(gain=5/3, use_tanh=True, use_batchnorm=False, normalize_fan_in=True):
    global g
    g = cpu_generator()
    C_deep = randn_on_device((vocab_size, n_embd), g)

    dims = [n_embd * block_size, 100, 100, 100, 100, 100, vocab_size]
    layers_deep = []
    for layer_index, (fan_in, fan_out) in enumerate(zip(dims, dims[1:])):
        linear = Linear(fan_in, fan_out, bias=not use_batchnorm)
        if not normalize_fan_in:
            with torch.no_grad():
                linear.weight.mul_(math.sqrt(fan_in))
        layers_deep.append(linear)
        if use_batchnorm:
            layers_deep.append(BatchNorm1d(fan_out))
        if use_tanh and layer_index < len(dims) - 2:
            layers_deep.append(Tanh())

    with torch.no_grad():
        if use_batchnorm:
            # last layer: make the softmax less confident
            layers_deep[-1].gamma.mul_(0.1)
            for layer in layers_deep[:-1]:
                if isinstance(layer, Linear):
                    layer.weight.mul_(gain)
        else:
            # last layer: make the softmax less confident
            layers_deep[-1].weight.mul_(0.1)
            for layer in layers_deep[:-1]:
                if isinstance(layer, Linear):
                    layer.weight.mul_(gain)

    parameters_deep = [C_deep] + [
        p for layer in layers_deep for p in layer.parameters()
    ]
    for p in parameters_deep:
        p.requires_grad_(True)
    return C_deep, layers_deep, parameters_deep


def set_layer_mode(layers_deep, training):
    for layer in layers_deep:
        if hasattr(layer, 'training'):
            layer.training = training


def run_deep_steps(C_deep, layers_deep, parameters_deep, steps, learning_rate=0.1, seed=SEED + 1):
    set_layer_mode(layers_deep, True)
    X_batches, Y_batches = prefetch_training_batches(steps, seed=seed)
    losses = torch.empty(steps, device=device)
    update_ratios = torch.empty((steps, len(parameters_deep)), device=device)

    for i in range(steps):
        Xb, Yb = X_batches[i], Y_batches[i]
        emb = C_deep[Xb]
        x = emb.view(emb.shape[0], -1)
        for layer in layers_deep:
            x = layer(x)
        loss = F.cross_entropy(x, Yb)

        # retain intermediate gradients only on the last step
        if i == steps - 1:
            for layer in layers_deep:
                layer.out.retain_grad()

        for p in parameters_deep:
            p.grad = None
        loss.backward()

        lr = learning_rate(i) if callable(learning_rate) else learning_rate
        with torch.no_grad():
            for j, p in enumerate(parameters_deep):
                denom = p.data.std()
                update_ratios[i, j] = ((lr * p.grad).std() / denom).log10()
                p.add_(p.grad, alpha=-lr)
            losses[i] = loss.log10()

        if i % max(1, steps // 10) == 0 or i == steps - 1:
            print(f'{i:5d}/{steps:5d}: {loss.item():.4f}')

    del X_batches, Y_batches
    return losses.detach().cpu(), update_ratios.detach().cpu()


def show_layer_histograms(layers_deep, layer_class, gradients=False, title=''):
    plt.figure(figsize=(16, 4))
    legends = []
    for i, layer in enumerate(layers_deep[:-1]):
        if isinstance(layer, layer_class):
            tensor = layer.out.grad if gradients else layer.out
            t = tensor.detach().float().cpu()
            if gradients:
                print(f'layer {i:2d} {layer.__class__.__name__:>11s}: grad mean {t.mean():+.2e}, std {t.std():.2e}')
            elif isinstance(layer, Tanh):
                saturation = (t.abs() > 0.97).float().mean() * 100
                print(f'layer {i:2d} {layer.__class__.__name__:>11s}: mean {t.mean():+.2f}, std {t.std():.2f}, saturated {saturation:.2f}%')
            else:
                print(f'layer {i:2d} {layer.__class__.__name__:>11s}: mean {t.mean():+.2f}, std {t.std():.2f}')
            hy, hx = torch.histogram(t, density=True)
            plt.plot(hx[:-1], hy)
            legends.append(f'layer {i}')
    plt.legend(legends)
    plt.title(title)
    plt.show()


def show_parameter_gradients(parameters_deep):
    plt.figure(figsize=(16, 4))
    legends = []
    for i, p in enumerate(parameters_deep):
        if p.ndim == 2:
            t = p.grad.detach().float().cpu()
            ratio = t.std() / p.detach().float().cpu().std()
            print(f'param {i:2d} {str(tuple(p.shape)):>12s} | grad mean {t.mean():+.2e} | grad std {t.std():.2e} | grad:data {ratio:.2e}')
            hy, hx = torch.histogram(t, density=True)
            plt.plot(hx[:-1], hy)
            legends.append(f'{i} {tuple(p.shape)}')
    plt.legend(legends)
    plt.title('parameter gradient distributions')
    plt.show()


def plot_update_ratios(update_ratios, parameters_deep, title):
    plt.figure(figsize=(16, 4))
    legends = []
    for i, p in enumerate(parameters_deep):
        if p.ndim == 2:
            plt.plot(update_ratios[:, i])
            legends.append(f'param {i}')
    plt.axhline(-3, color='black', linewidth=1)
    plt.legend(legends)
    plt.ylabel('log10(update std / parameter std)')
    plt.xlabel('step')
    plt.title(title + ' (rough guide: -3)')
    plt.show()


In [ ]:
C_deep, layers_deep, parameters_deep = build_deep_network(
    gain=5/3, use_tanh=True, use_batchnorm=False, normalize_fan_in=True
)
print('parameter count:', sum(p.numel() for p in parameters_deep))
deep_losses, deep_ud = run_deep_steps(
    C_deep, layers_deep, parameters_deep, steps=1, learning_rate=0.1
)


In [ ]:
show_layer_histograms(
    layers_deep, Tanh, gradients=False,
    title='forward activations: tanh gain = 5/3'
)
show_layer_histograms(
    layers_deep, Tanh, gradients=True,
    title='backward gradients: tanh gain = 5/3'
)


In [ ]:
C_deep, layers_deep, parameters_deep = build_deep_network(
    gain=1.0, use_tanh=True, use_batchnorm=False, normalize_fan_in=True
)
deep_losses, deep_ud = run_deep_steps(C_deep, layers_deep, parameters_deep, 1, 0.1)
show_layer_histograms(layers_deep, Tanh, False, 'forward activations: gain = 1.0')
show_layer_histograms(layers_deep, Tanh, True, 'backward gradients: gain = 1.0')


In [ ]:
C_deep, layers_deep, parameters_deep = build_deep_network(
    gain=3.0, use_tanh=True, use_batchnorm=False, normalize_fan_in=True
)
deep_losses, deep_ud = run_deep_steps(C_deep, layers_deep, parameters_deep, 1, 0.1)
show_layer_histograms(layers_deep, Tanh, False, 'forward activations: gain = 3.0')
show_layer_histograms(layers_deep, Tanh, True, 'backward gradients: gain = 3.0')


In [ ]:
C_deep, layers_deep, parameters_deep = build_deep_network(
    gain=5/3, use_tanh=False, use_batchnorm=False, normalize_fan_in=True
)
deep_losses, deep_ud = run_deep_steps(C_deep, layers_deep, parameters_deep, 1, 0.1)
show_layer_histograms(layers_deep, Linear, False, 'deep linear network: gain = 5/3')
show_layer_histograms(layers_deep, Linear, True, 'deep linear gradients: gain = 5/3')


In [ ]:
C_deep, layers_deep, parameters_deep = build_deep_network(
    gain=1.0, use_tanh=False, use_batchnorm=False, normalize_fan_in=True
)
deep_losses, deep_ud = run_deep_steps(C_deep, layers_deep, parameters_deep, 1, 0.1)
show_layer_histograms(layers_deep, Linear, False, 'deep linear network: gain = 1.0')
show_layer_histograms(layers_deep, Linear, True, 'deep linear gradients: gain = 1.0')


In [ ]:
C_deep, layers_deep, parameters_deep = build_deep_network(
    gain=5/3, use_tanh=True, use_batchnorm=False, normalize_fan_in=True
)
deep_losses, deep_ud = run_deep_steps(
    C_deep, layers_deep, parameters_deep, DEBUG_STEPS, learning_rate=0.1
)
show_layer_histograms(layers_deep, Tanh, False, 'activations after 1000 steps')
show_layer_histograms(layers_deep, Tanh, True, 'gradients after 1000 steps')
show_parameter_gradients(parameters_deep)


In [ ]:
plot_update_ratios(deep_ud, parameters_deep, 'update:data, lr = 0.1')


In [ ]:
C_deep, layers_deep, parameters_deep = build_deep_network(
    gain=5/3, use_tanh=True, use_batchnorm=False, normalize_fan_in=True
)
low_lr_losses, low_lr_ud = run_deep_steps(
    C_deep, layers_deep, parameters_deep, DEBUG_STEPS, learning_rate=0.001
)
plot_update_ratios(low_lr_ud, parameters_deep, 'update:data, lr = 0.001')


In [ ]:
C_deep, layers_deep, parameters_deep = build_deep_network(
    gain=5/3, use_tanh=True, use_batchnorm=False, normalize_fan_in=False
)
bad_init_losses, bad_init_ud = run_deep_steps(
    C_deep, layers_deep, parameters_deep, DEBUG_STEPS, learning_rate=0.1
)
show_layer_histograms(layers_deep, Tanh, False, 'bad init: activations')
show_layer_histograms(layers_deep, Tanh, True, 'bad init: gradients')
show_parameter_gradients(parameters_deep)
plot_update_ratios(bad_init_ud, parameters_deep, 'bad init: update:data')


## batch norm in the deeper network

Insert BatchNorm between each Linear layer and Tanh. The activations become much less sensitive to the weight gain, but the update scale still changes.


In [ ]:
C_deep, layers_deep, parameters_deep = build_deep_network(
    gain=5/3, use_tanh=True, use_batchnorm=True, normalize_fan_in=True
)
bn_losses, bn_ud = run_deep_steps(
    C_deep, layers_deep, parameters_deep, DEBUG_STEPS, learning_rate=0.1
)
show_layer_histograms(layers_deep, Tanh, False, 'BatchNorm: forward activations')
show_layer_histograms(layers_deep, Tanh, True, 'BatchNorm: backward gradients')
show_parameter_gradients(parameters_deep)
plot_update_ratios(bn_ud, parameters_deep, 'BatchNorm, gain = 5/3, lr = 0.1')


In [ ]:
C_deep, layers_deep, parameters_deep = build_deep_network(
    gain=2.0, use_tanh=True, use_batchnorm=True, normalize_fan_in=True
)
bn_gain2_losses, bn_gain2_ud = run_deep_steps(
    C_deep, layers_deep, parameters_deep, DEBUG_STEPS, learning_rate=0.1
)
show_layer_histograms(layers_deep, Tanh, False, 'BatchNorm: gain = 2.0')
show_layer_histograms(layers_deep, Tanh, True, 'BatchNorm gradients: gain = 2.0')
plot_update_ratios(bn_gain2_ud, parameters_deep, 'BatchNorm, gain = 2.0, lr = 0.1')


In [ ]:
C_deep, layers_deep, parameters_deep = build_deep_network(
    gain=1.0, use_tanh=True, use_batchnorm=True, normalize_fan_in=False
)
bn_raw_losses, bn_raw_ud = run_deep_steps(
    C_deep, layers_deep, parameters_deep, DEBUG_STEPS, learning_rate=0.1
)
show_layer_histograms(layers_deep, Tanh, False, 'BatchNorm without fan-in scaling')
show_layer_histograms(layers_deep, Tanh, True, 'BatchNorm gradients without fan-in scaling')
plot_update_ratios(bn_raw_ud, parameters_deep, 'no fan-in scaling, lr = 0.1')


In [ ]:
C_deep, layers_deep, parameters_deep = build_deep_network(
    gain=1.0, use_tanh=True, use_batchnorm=True, normalize_fan_in=False
)
bn_lr1_losses, bn_lr1_ud = run_deep_steps(
    C_deep, layers_deep, parameters_deep, DEBUG_STEPS, learning_rate=1.0
)
show_layer_histograms(layers_deep, Tanh, False, 'BatchNorm, no fan-in, lr = 1.0')
show_layer_histograms(layers_deep, Tanh, True, 'BatchNorm gradients, lr = 1.0')
plot_update_ratios(bn_lr1_ud, parameters_deep, 'no fan-in scaling, lr = 1.0')


In [ ]:
@torch.no_grad()
def evaluate_deep(C_deep, layers_deep):
    set_layer_mode(layers_deep, False)
    for name, x, y in (('train', Xtr, Ytr), ('val', Xdev, Ydev), ('test', Xte, Yte)):
        out = C_deep[x].view(x.shape[0], -1)
        for layer in layers_deep:
            out = layer(out)
        print(f'{name:>5s}: {F.cross_entropy(out, y).item():.6f}')

evaluate_deep(C_deep, layers_deep)


In [ ]:
@torch.no_grad()
def sample_deep(C_deep, layers_deep, count=20):
    set_layer_mode(layers_deep, False)
    generator_device = 'cuda' if device.type == 'cuda' else 'cpu'
    sg = torch.Generator(device=generator_device).manual_seed(SEED + 10)
    for _ in range(count):
        out_chars = []
        context = [0] * block_size
        while True:
            x = torch.tensor([context], dtype=torch.long, device=device)
            out = C_deep[x].view(1, -1)
            for layer in layers_deep:
                out = layer(out)
            probs = F.softmax(out, dim=1)
            ix = torch.multinomial(probs, 1, generator=sg).item()
            context = context[1:] + [ix]
            out_chars.append(ix)
            if ix == 0:
                break
        print(''.join(itos[i] for i in out_chars))

sample_deep(C_deep, layers_deep)


## loss log

### original
train 2.1245384216308594  
val 2.168196439743042

### fix softmax confidently wrong
train ~2.07  
val ~2.13

### fix tanh layer too saturated at init
train 2.0355966091156006  
val 2.1026785373687744

### use semi-principled "kaiming init" instead of hacky init
train 2.0376641750335693  
val 2.106989622116089

### add batch norm layer
train 2.0668270587921143  
val 2.104844808578491
